In [ ]:
pip install torch==2.9.0+cu126 --extra-index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu126


In [ ]:
# Modelin sığması için quantization (bitsandbytes) ve hızlandırma (accelerate) kütüphaneleri şart
!pip install -q -U torch transformers datasets peft bitsandbytes trl accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-v0.1"

# 1. 4-Bit Sıkıştırma Ayarları
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Tokenizer'ı Yükle
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Hata almamak için padding token'ı tanımlıyoruz

# 3. Modeli Yükle
print("Model indiriliyor ve yükleniyor... Lütfen bekleyin.")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model başarıyla yüklendi!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model başarıyla yüklendi!


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)

!pip install -q trl # Install the missing trl library
from trl import SFTTrainer

# ---------------------------------------------------------
# 1. AYARLAR (Configuration)
# ---------------------------------------------------------
model_name = "mistralai/Mistral-7B-v0.1"
new_model_name = "mistral-7b-turkce-duzeltici" # Eğitilen modelin yeni adı

# QLoRA Ayarları (Modelin %1'ini eğitmek için)
peft_config = LoraConfig(
    r=16,       # Rank (Eğitilebilir parametre sayısı, 8, 16, 64 olabilir)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] # Mistral'in dikkat mekanizmaları
)

# 4-Bit Yükleme Ayarları (Hafıza tasarrufu için)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# ---------------------------------------------------------
# 2. MODEL VE TOKENIZER YÜKLEME
# ---------------------------------------------------------
print("Model yükleniyor...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False # Eğitim sırasında cache kapatılır
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Padding hatasını önlemek için
tokenizer.padding_side = "right"

# Modeli eğitime hazırla
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

# ---------------------------------------------------------
# 3. VERİ SETİNİ YÜKLEME VE FORMATLAMA
# ---------------------------------------------------------
print("Veri seti yükleniyor...")
dataset = load_dataset("json", data_files="ham_veri_seti.json", split="train")

# Modele veriyi nasıl sunacağımızı belirleyen fonksiyon
# Prompt Formatı: Instruction + Input -> Response
def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['prompt'])):
        text = f"### Instruction:\nAşağıdaki bozuk Türkçe metni düzelt ve akıcı hale getir.\n\n### Input:\n{example['raw_model_output'][i]}\n\n### Response:\n{example['target_text'][i]}"
        output_texts.append(text)
    return output_texts

# ---------------------------------------------------------
# 4. EĞİTİMİ BAŞLATMA (TRAINING)
# ---------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,          # Veri setini kaç tur döneceği (Verin az ise 3-5 yapabilirsin)
    per_device_train_batch_size=4, # RAM yetmezse 2'ye düşür
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1, # -1 bırakırsan epochs kadar döner
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
)

print("Eğitim başlıyor... Arkana yaslan!")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=None,
    dataset_text_field=None, # formatting_func kullandığımız için None
    formatting_func=formatting_prompts_func,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

trainer.train()

# ---------------------------------------------------------
# 5. MODELİ KAYDETME
# ---------------------------------------------------------
print("Eğitim bitti. Model kaydediliyor...")
trainer.model.save_pretrained(new_model_name)
print(f"Model '{new_model_name}' klasörüne kaydedildi.")

Model yükleniyor...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Veri seti yükleniyor...


FileNotFoundError: Unable to find '/content/ham_veri_seti.json'

In [ ]:
# Modelin tamamlaması için bir başlangıç cümlesi (Prompt) veriyoruz
input_text = "nane yemek en çok"

# Metni sayılara (vektörlere) çevir
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

# Modeli çalıştır (Generation)
outputs = model.generate(
    **inputs,
    max_new_tokens=200,  # En fazla 100 yeni kelime/token üretsin
    temperature=0.7,     # Yaratıcılık seviyesi (0.1 robotik, 1.0 çok rastgele)
    do_sample=True,      # Rastgele örnekleme yap
    repetition_penalty=1.2 # Sürekli aynı şeyi tekrarlamasını engelle
)

# Çıktıyı tekrar yazıya çevir
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("-" * 30)
print(f"GİRİŞ: {input_text}")
print("-" * 30)
print(f"ÇIKTI:\n{generated_text}")
print("-" * 30)

In [ ]:
import json
from tqdm import tqdm # İlerleme çubuğu görmek için

# 1. ADIM: Başlangıç Cümleleri (Prompts) Hazırlama
# Buraya ne kadar çok çeşit eklersen, veri setin o kadar zengin olur.
prompts = [
    # Teknik / Yazılım Konuları (Genelde İngilizce karışık bozuk Türkçe verir)
    "Python programlama dilinde bir liste oluşturmak için",
    "Yapay sinir ağlarında backpropagation algoritması çalışırken",
    "SQL veritabanında tablo birleştirmek için join komutu",
    "React kütüphanesinde component yapısı şöyledir",
    "Linux işletim sisteminde terminal komutları kullanırken",

    # Günlük Hayat / Genel Kültür (Devrik cümle kurma ihtimali yüksek)
    "İstanbul'da trafiğin en yoğun olduğu saatlerde",
    "Sürdürülebilir enerji kaynaklarının kullanımı dünyada",
    "Kahve yaparken dikkat edilmesi gereken en önemli nokta",
    "Osmanlı İmparatorluğu'nun yükselme döneminde",
    "Futbolda ofsayt kuralının mantığı aslında",

    # Akademik / Resmi Dil (Karmaşık cümle yapıları)
    "Küresel ısınmanın ekonomik etkileri incelendiğinde",
    "Makine öğrenmesi algoritmalarının etik problemleri",
    "Modern fizik teorilerine göre zaman kavramı",

    # Yazılım
    "Kod derlenirken hata verince",
    "Sunucuya bağlantı koptuğunda",
    "Veritabanı sorgusu gecikince",
    "Python döngüsü sonsuza girdiğinde",
    "API anahtarı geçersiz olunca",
    "Git merge conflict yaşanırken",
    "Yeni güncelleme yayınlandığında",
    "Kullanıcı arayüzü donduğunda",
    "Yapay zeka modeli eğitilirken",
    "Değişken tanımlanmadan kullanıldığında",
    "CSS dosyası yüklenmeyince",
    "Güvenlik duvarı engellediğinde",
    "Mobil uygulama çöktüğünde",
    "Docker konteyneri başlatılırken",
    "Yedekleme işlemi tamamlanınca",

    # Tarih
    "Roma İmparatorluğu yıkılmaya yüz tutunca",
    "Savaşın seyrini değiştiren o hamleyle",
    "Padişah fermanı imzaladıktan sonra",
    "Sanayi devrimi başladığında",
    "Berlin Duvarı yıkılırken",
    "Antik kentteki kazılar sırasında",
    "Fransız İhtilali'nin etkisiyle",
    "Barış antlaşması masada kalınca",
    "Viking gemileri ufukta görününce",
    "Soğuk Savaş döneminde",
    "Mısır piramitleri inşa edilirken",
    "Orta Çağ karanlığında",
    "Fetih hazırlıkları sürerken",
    "Bağımsızlık bildirgesi okunduğunda",
    "İpek Yolu üzerindeki kervanlar",

    # Spor
    "Hakem son düdüğü çalınca",
    "Doksana giden topu kaleci çıkarınca",
    "Maçın uzatma dakikalarında",
    "Maratonun son kilometresine girerken",
    "Penaltı atışını kullanmak için",
    "Takım kaptanı sakatlanınca",
    "Ofsayt bayrağı havaya kalktığında",
    "Basketbol potası kırılınca",
    "Tenis maçında servis atarken",
    "Formula 1 aracı pit stopa girince",
    "Tribünler hep bir ağızdan bağırırken",
    "Olimpiyat meşalesi yakıldığında",
    "Sarı kart gören oyuncu",
    "Rakip savunma hattını yararken",
    "Şampiyonluk kupası kaldırılırken",

    # Mutfak & Yemek
    "Soğanlar pembeleşinceye kadar",
    "Yemeğin tuzu fazla kaçınca",
    "Fırının derecesini ayarlarken",
    "Hamur mayalanmaya bırakıldığında",
    "Sos kıvam almaya başlayınca",
    "Tencerenin dibi tuttuğunda",
    "Bıçağın keskin tarafıyla",
    "Yumurtaları köpürene kadar çırparken",
    "Makarna suyu kaynadığında",
    "Kekin içi çiğ kalınca",
    "Baharatları eklemeden önce",
    "Sütü ocakta unutunca",
    "Sebzeleri ince ince doğrayınca",
    "Tatlı şerbetini çekerken",
    "Izgaradaki etler cızırdarken",

    # Ekonomi
    "Enflasyon oranları açıklandığında",
    "Dolar kuru aniden yükselince",
    "Borsa günü düşüşle kapatınca",
    "Merkez Bankası faiz kararını verince",
    "Yatırımcılar paniğe kapıldığında",
    "Kripto para piyasası çökünce",
    "Asgari ücret zammı konuşulurken",
    "İthalat vergileri artırıldığında",
    "Şirket hisseleri değer kazanınca",
    "Altın fiyatları rekor kırarken",
    "Bütçe açığı büyüdüğünde",
    "Kredi notu düşürülünce",
    "Ekonomik durgunluk başladığında",
    "Vergi yapılandırması çıkınca",
    "Petrol fiyatları dalgalanırken",

    # Bilim
    "Atom parçalandığı anda",
    "DNA sarmalı incelendiğinde",
    "Kara delik fotoğraflandığında",
    "Deney tüpündeki reaksiyon başlayınca",
    "Işık hızına yaklaşıldığında",
    "Mars'ta su bulunduğu haberiyle",
    "Hücre bölünmesi sırasında",
    "Kuantum fiziğine göre",
    "Laboratuvar sonuçları çıkınca",
    "Yerçekimsiz ortamda hareket ederken",
    "Teleskopla gökyüzüne bakınca",
    "Bakteriler hızla çoğaldığında",
    "Aşı çalışmaları tamamlanınca",
    "Volkanik patlama öncesinde",
    "Buzullar erimeye başladığında",

    # Günlük Hayat
    "Sabah alarmı çalmadığında",
    "Otobüse yetişmek için koşarken",
    "Anahtarımı evde unuttuğumda",
    "Kahvemi yudumlarken",
    "Trafik sıkışıklığı yüzünden",
    "Market alışverişi yaparken",
    "Telefonun şarjı bitince",
    "Yağmur aniden bastırınca",
    "Asansör bozulduğunda",
    "Komşudan gürültü gelince",
    "Kargo paketi gelmeyince",
    "Gömleğime kahve dökülünce",
    "Cüzdanımı bulamayınca",
    "İnternet bağlantısı gidince",
    "Ayakkabımın bağcığı çözülünce",

    # Yazılım
    "Deploy işlemi başarısız olunca",
    "Log dosyaları diski doldurunca",
    "Sanal makine yanıt vermediğinde",
    "Algoritma optimize edilirken",
    "SQL injection saldırısı gelince",
    "Kütüphane versiyonu uyuşmayınca",
    "Mavi ekran hatası alındığında",
    "Tarayıcı önbelleği temizlenmeyince",
    "API istek limiti aşılınca",
    "Token süresi dolduğunda",
    "Recursive fonksiyon hatasıyla",
    "Karanlık mod aktifleşince",
    "GitHub reposu silinince",
    "Yazılım güncellemesi yarım kalınca",
    "Kod incelemesi sırasında",

    # Tarih
    "Matbaa ilk kez çalıştırıldığında",
    "Rönesans sanatçıları eser verirken",
    "Truva atı şehre girince",
    "Berlin Duvarı'na tırmanırken",
    "Ay'a ilk ayak basıldığında",
    "Titanik buzdağına çarpınca",
    "Çin Seddi örülürken",
    "Magna Carta imzalandığında",
    "Moğol orduları ufukta belirince",
    "İstanbul kuşatması sırasında",
    "Büyük Buhran patlak verince",
    "Gladyatör arenaya çıkınca",
    "Kral tacını takarken",
    "Colosseum inşaatında",
    "Veba salgını şehre yayılınca",

    # Spor
    "VAR incelemesi yapılırken",
    "Teknik direktör sahaya girince",
    "Yedek kulübesi karışınca",
    "Son saniye basketi potadan dönünce",
    "Depar atarken adelesi çekince",
    "Boksör ringde sendeleyince",
    "Yüzücü havuza atladığında",
    "Fileler havalandığı anda",
    "Hatalı çıkış yapılınca",
    "Kırmızı kart havaya kalkınca",
    "Voleybolda blok yaparken",
    "Ralli aracı virajı alamayınca",
    "Golf topu deliğin kenarında durunca",
    "Kaleci degaj yaparken",
    "Taraftar sahaya atlayınca",

    # Mutfak & Yemek
    "Krep tavaya yapıştığında",
    "Çaydanlık kireç tutunca",
    "Buzluktan kıyma çıkarınca",
    "Patatesler kızgın yağa atılınca",
    "Fırın sütlaç üzerini yakarken",
    "Mayalı hamur taştığında",
    "Blender bozulup durunca",
    "Döküm tava çok ısınınca",
    "Limona bıçak değince",
    "Türk kahvesi taşınca",
    "Ekmekler fırında kuruyunca",
    "Balık ayıklarken",
    "Közde biber kokusu gelince",
    "Turşu kavanozu sıkışınca",
    "Reçel şekerlenmeye başlayınca",

    # Ekonomi
    "Kredi kartı limiti dolduğunda",
    "Banka hesabı bloke olunca",
    "Hisse senetleri çakıldığında",
    "Konut fiyatları tavan yapınca",
    "Turizm gelirleri beklenmeyince",
    "Yatırım fonu değer kazandığında",
    "Bitcoin cüzdan şifresi unutulunca",
    "İhracat rekoru kırıldığında",
    "Faiz indirimi kararı gelince",
    "Borsa İstanbul açılış yapınca",
    "Tahvil faizleri yükselince",
    "Şirket birleşmesi duyurulunca",
    "Vergi rekortmenleri açıklanırken",
    "Start-up yatırımı alınca",
    "Banka kredisi onaylanmayınca",

    # Bilim
    "Güneş tutulması izlenirken",
    "Deprem dalgaları sismografa yansıyınca",
    "Fosil yakıtlar tükenmeye başlayınca",
    "Yapay zeka kendi kendine kod yazınca",
    "Robotlar fabrikada iş başı yapınca",
    "Ozon tabakasındaki delik büyüyünce",
    "Genetik mutasyon tespit edildiğinde",
    "Nötron yıldızları çarpışınca",
    "Süper iletken bulunduğunda",
    "Maddenin dördüncü hali plazmada",
    "Kimyasal tepkime sona erince",
    "Astronot kaskını taktığında",
    "Meteor yağmuru başlarken",
    "Virüs laboratuvar ortamında",
    "Kök hücre tedavisi denenirken",

    # Günlük Hayat
    "Kedi mama kabını devirince",
    "Komşunun tadilat sesi başlayınca",
    "Elektrikler aniden kesilince",
    "Su faturası yüksek gelince",
    "Kargocu evde kimseyi bulamayınca",
    "Kombinin basıncı düşünce",
    "Ütü yaparken gömlek yanınca",
    "Saç kurutma makinesi alev alınca",
    "Metro kapısına sıkışırken",
    "Sinemada mısır dökülünce",
    "Kulaklık kablosu düğüm olunca",
    "Sivrisinek kulağımda vızıldayınca",
    "Çalar saat ısrarla çalınca",
    "E-posta şifremi unutunca",
    "Diş macunu tüpü bitince",

    # Yazılım
    "Debug modunda adım adım ilerlerken",
    "JSON verisi parse edilemeyince",
    "Linux terminalinde root yetkisiyle",
    "Veri sızıntısı tespit edildiğinde",
    "Siber güvenlik saldırısı sırasında",
    "Cache belleği temizlemeyi unutunca",
    "Nesne tabanlı programlamada",
    "Responsive tasarım mobilde bozulunca",
    "Bulut sunucuya dosya yüklerken",
    "404 hata sayfası ekrana gelince",
    "Yazılım lisansı sona erdiğinde",
    "Açık kaynak koda katkı yaparken",
    "Binary kodlarını incelerken",
    "Yapay sinir ağları eğitilirken",
    "Veritabanı bağlantı havuzu dolunca",

    # Tarih
    "Sümer tabletleri tercüme edilirken",
    "Kristof Kolomb karayı gördüğünde",
    "Piri Reis haritayı çizerken",
    "Çanakkale Boğazı geçilmeye çalışılınca",
    "Hitler sığınağına çekildiğinde",
    "Bastille Hapishanesi basıldığında",
    "Osmanlı donanması sefere çıkarken",
    "Atatürk Samsun'a ayak bastığında",
    "Pompei Yanardağı patladığı anda",
    "İskenderiye Kütüphanesi yanarken",
    "Soğuk Savaş'ın en gergin anında",
    "Sanayi devrimi fabrikalarında",
    "Demir Perde ülkelerinde",
    "Viyana kapılarına dayanılınca",
    "Tekerlek icat edildiğinde",

    # Spor
    "Frikik barajı kurulurken",
    "Tenis topu çizgiye değince",
    "Basketbol koçu mola aldığında",
    "100 metre finali başlarken",
    "Formula 1 damalı bayrağı sallanınca",
    "Sakatlanan oyuncu sedyeyle çıkarken",
    "Voleybol maçı tie-break setine gidince",
    "Boks maçının son raundunda",
    "Golf sahasındaki derin sessizlikte",
    "Şampiyonlar Ligi müziği çalınca",
    "Bisiklet turunda yokuş tırmanırken",
    "Hakem VAR monitörüne gidince",
    "Transfer dönemi kapanmadan önce",
    "Doping testi pozitif çıkınca",
    "Stadyum ışıkları sönünce",

    # Mutfak & Yemek
    "Patatesler nar gibi kızarınca",
    "Mantı kaynar suya atıldığında",
    "Karpuzun içi geçmiş çıkınca",
    "Çorbanın tuzu eksik kalınca",
    "Karnıyarık fırına verilince",
    "Yoğurt mayalanmaya bırakıldığında",
    "Sufle sönmeden servis edilince",
    "Mangal kömürü kor haline gelince",
    "Zeytinyağı tavada ısınınca",
    "Hamur merdane ile açılırken",
    "Baklava şerbeti dökülünce",
    "Balık ızgaraya yapışınca",
    "Pazar kahvaltısı hazırlanırken",
    "Sarımsak ezilirken çıkan kokuyla",
    "Pilav tane tane dökülünce",

    # Ekonomi
    "Döviz bürosu önünde kuyruk olunca",
    "Merkez Bankası rezervleri eriyince",
    "Konkordato ilan eden şirketler",
    "Kredi kartı borcu yapılandırılınca",
    "Emekli maaşlarına zam gelince",
    "Petrol varil fiyatı aniden düşünce",
    "Borsa spekülasyonları artınca",
    "Enflasyon sepeti değiştiğinde",
    "İşsizlik rakamları açıklanınca",
    "Turist sayısı beklentiyi aşınca",
    "İhracat teşvikleri verilince",
    "Altın rezervleri artırıldığında",
    "Faizsiz kredi kampanyası başlayınca",
    "Ekonomik kriz teğet geçince",
    "Maaş bordrosunu incelediğimde",

    # Bilim
    "Mikroskop altında bakteri incelenirken",
    "Hubble teleskobu yeni galaksi bulunca",
    "Periyodik tabloya yeni element eklenince",
    "Füzyon enerjisi elde edildiğinde",
    "Yapay zeka Turing testini geçince",
    "Küresel ısınma raporu yayınlanınca",
    "Kara madde hakkında teori üretirken",
    "Göktaşı dünyaya tehlikeli yaklaşınca",
    "Klonlama deneyi başarılı olunca",
    "Mars kolonisi kurulduğunda",
    "Kuantum bilgisayar şifreyi çözünce",
    "Antarktika buzulları erirken",
    "Elektrikli araç bataryası geliştirilirken",
    "Nanoteknoloji tıpta kullanılınca",
    "Uluslararası Uzay İstasyonu'nda",

    # Günlük Hayat
    "Otobüs kartımda bakiye bitince",
    "Apartman aidatı gecikince",
    "Dişçi randevusunu beklerken",
    "Kışlık kıyafetleri hurçtan çıkarırken",
    "Balkonda çay keyfi yaparken",
    "Sokak kedisine mama verirken",
    "Trafik lambası bozulunca",
    "ATM paramı yutunca",
    "Misafirliğe giderken alınan hediye",
    "Pazar sabahı sessizliğinde",
    "Piknik tüpü bitince",
    "Akbil doldurma sırasındayken",
    "Kombinin su basıncını ayarlarken",
    "Market poşeti yolda yırtılınca",
    "Sınav sonuçları açıklanırken",

    # --- YAZILIM VE TEKNOLOJİ (25 Adet) ---
    "Stack Overflow'da çözüm ararken",
    "Sonsuz döngüden çıkamayınca",
    "Veri merkezi elektrik kesintisi yaşayınca",
    "Büyük veri analizi tamamlandığında",
    "Yapay zeka halüsinasyon görünce",
    "Blockchain zinciri senkronize olurken",
    "Sanal gerçeklik gözlüğünü taktığımda",
    "5G teknolojisi yaygınlaşınca",
    "Klavye üzerine kahve dökülünce",
    "Sunucu yanıt süresi uzadığında",
    "Spagetti koda müdahale ederken",
    "Admin şifresini sıfırladıktan sonra",
    "CSS Grid yapısı bozulunca",
    "Python kütüphanesi yüklenmeyince",
    "Ransomware virüsü dosyaları şifreleyince",
    "Bulut tabanlı yedekleme duraklatılınca",
    "Akıllı saatim titremeye başlayınca",
    "Kod refactoring işlemi sırasında",
    "Yazılımın beta sürümü yayınlanınca",
    "Frontend ve backend uyumsuz olunca",
    "Veritabanı tablosu silindiğinde",
    "Nesnelerin interneti (IoT) cihazları",
    "Biyometrik doğrulama başarısız olunca",
    "Derleyici optimizasyon yaparken",
    "Açık kaynak komünitesi destek verince",

    # --- TARİH (25 Adet) ---
    "Sanayi Devrimi'nin buharlı makineleri",
    "Telgraf hatları ilk kez çekildiğinde",
    "Süveyş Kanalı açıldığı gün",
    "Berlin Antlaşması masasında",
    "Vikingler İngiltere kıyılarına inince",
    "Samuray kılıcını çektiğinde",
    "Aztek tapınaklarında ayin yapılırken",
    "Soğuk Savaş casusları buluştuğunda",
    "Wright kardeşler ilk uçuşu yaparken",
    "Matbaada ilk gazete basıldığında",
    "Altına Hücum dönemi başladığında",
    "İpek Yolu tüccarları dinlenirken",
    "Truva Savaşı'nın onuncu yılında",
    "Napolyon sürgüne gönderilince",
    "Hiroşima'ya atom bombası düştüğünde",
    "Uzay yarışı hız kazandığında",
    "Mimar Sinan camiyi tasarlarken",
    "Çernobil reaktörü patlamadan önce",
    "Osmanlı'da Lale Devri yaşanırken",
    "Magellan dünyayı dolaşmaya çıkınca",
    "Bolşevik İhtilali sokaklara taşınca",
    "Haçlı Seferleri kudüs yolunda",
    "Nil Nehri taştığında Mısırlılar",
    "Roma Senatosu toplandığında",
    "Göbeklitepe sütunları dikilirken",

    # --- SPOR (25 Adet) ---
    "Maratonun son düzlüğüne girerken",
    "Tenis raketinin telleri kopunca",
    "Basketbolda smaç basıldığında",
    "Yüzücü takla atıp dönerken",
    "Formula 1 pilotu virajı geniş alınca",
    "Kaleci ters ayakta yakalanınca",
    "Satrançta şah mat hamlesi gelince",
    "Güreşçi rakibini tuş edince",
    "Okçulukta hedef tam 12'den vurulunca",
    "Buz patencisi dengesini kaybedince",
    "Hakem düdüğünü ağzına götürürken",
    "Deplasman tribünü sessizliğe gömülünce",
    "Transfer dedikoduları alevlenince",
    "Sakatlıktan dönen oyuncu gol atınca",
    "Halterci ağırlığı omuzladığında",
    "Bilardo topu banda çarpınca",
    "Kayakçı pistten aşağı süzülürken",
    "Olimpiyat köyünde heyecan artınca",
    "Derbi maçı öncesi atmosfer",
    "Korner bayrağının orada top saklarken",
    "E-spor turnuvası final maçında",
    "Rakip takım pres yapmaya başlayınca",
    "Teknik direktör ceketini çıkarınca",
    "Milli marş okunurken stadyumda",
    "Şampiyonluk turu atılırken",

    # --- MUTFAK VE YEMEK (25 Adet) ---
    "Karnabahar haşlanırken çıkan koku",
    "Tost makinesi duman çıkarınca",
    "Limonata yaparken şekeri az gelince",
    "Mangal ateşini yellemeye çalışırken",
    "Suşi pirinci lapa olunca",
    "Fırından taze ekmek kokusu gelince",
    "Bulaşık makinesi su akıttığında",
    "Dolma biberlerin içi doldurulurken",
    "Şerbetli tatlının üzerine fıstık dökünce",
    "Künefe peyniri uzadıkça uzarken",
    "Sahur sofrası hazırlanırken",
    "İftar topu patlamak üzereyken",
    "Kavanoz kapağı bir türlü açılmayınca",
    "Tavuk suyuna çorba kaynarken",
    "Acı biber dilimi yakınca",
    "Pizza hamuru havada çevrilirken",
    "Dondurma külahı erimeye başlayınca",
    "Patlamış mısır tencereden taşınca",
    "Sütlü kahvenin köpüğü sönünce",
    "Tuzlu kurabiyeler fırına girince",
    "Salata sosunu çalkalarken",
    "Kurban bayramı kavurması yapılırken",
    "Balık pazarında taze hamsi seçerken",
    "Gözleme sacı ısındığında",
    "Çikolata benmari usulü eritilirken",

    # --- EKONOMİ (25 Adet) ---
    "Kripto borsasında boğa sezonu başlayınca",
    "Startup şirketi yatırım turuna çıkınca",
    "Vergi dairesinden tebligat gelince",
    "Maaş zammı beklentinin altında kalınca",
    "Bireysel emeklilik fonu değerlenince",
    "Merkez Bankası rezerv opsiyonu kullanınca",
    "Tüketici güven endeksi düşünce",
    "Holding CEO'su istifa edince",
    "İkinci el araba piyasası durgunlaşınca",
    "Altın bilezik bozdurmaya gidince",
    "Kredi notunu yükseltmek için",
    "İcra memurları kapıya gelince",
    "Fuar alanında pazarlık yapılırken",
    "Stagflasyon riski konuşulurken",
    "Borsa çöküşü manşetlere taşınınca",
    "Asgari ücret tespit komisyonu toplanınca",
    "Döviz kurları ekranı kıpkırmızı olunca",
    "Gayrimenkul yatırımı değerlenince",
    "Halka arz talebi toplanırken",
    "Pazarda sebze fiyatları el yakınca",
    "Fatura ödeme tarihi geçince",
    "Temettü ödemesi hesaba yatınca",
    "Kayıt dışı ekonomiyle mücadele edilirken",
    "Global piyasalar resesyona girince",
    "Petrol zengini ülkeler vanayı kısınca",

    # --- BİLİM (25 Adet) ---
    "Fotosentez yapan bitkiler incelenirken",
    "Newton başına elma düştüğünde",
    "Arşimet hamamdan fırladığında",
    "Marie Curie laboratuvarında çalışırken",
    "Mars yüzeyine inen araçtan ilk sinyal gelince",
    "Kara deliğin olay ufkuna yaklaşırken",
    "Kutup ışıkları gökyüzünü aydınlatınca",
    "Yanardağ lav püskürtmeye başladığında",
    "Tsunami dalgaları kıyıya yaklaşınca",
    "DNA testi sonuçları beklenirken",
    "Kuantum dolanıklığı ispatlanınca",
    "Büyük Patlama (Big Bang) teorisine göre",
    "Antibiyotik direnci gelişen bakteriler",
    "Güneş panelleri enerji depolarken",
    "Rüzgar türbinleri dönmeye başlayınca",
    "Sanal gerçeklikte beyin dalgaları",
    "Yapay organ nakli gerçekleştiğinde",
    "Nesli tükenen hayvanlar klonlanınca",
    "Süpernovalar patladığında",
    "Yer kabuğu hareket etmeye başlayınca",
    "Atmosfere giren meteor yanarken",
    "Derin okyanus çukurlarına inilince",
    "Parçacık hızlandırıcı çalıştırıldığında",
    "Evrim teorisi tartışılırken",
    "Geleceğe zaman yolculuğu mümkün olsaydı",

    # --- GÜNLÜK HAYAT (25 Adet) ---
    "Metroda boş koltuk bulduğumda",
    "Sabah trafiğinde köprüde kalınca",
    "Yağmurlu havada taksi bulamayınca",
    "Pazar günü öğlene kadar uyuyunca",
    "Kuaför saçımı çok kısa kesince",
    "Terzi pantolon paçasını kıvırırken",
    "Kargo takip numarası hatalı çıkınca",
    "Market kasasında bozuk para ararken",
    "Komşunun köpeği havlamaya başlayınca",
    "Apartman toplantısında kavga çıkınca",
    "Çamaşır makinesi sıkma yaparken",
    "Ütü masasını kurmaya çalışırken",
    "Balkona çamaşır asarken yağmur başlayınca",
    "Televizyon kumandasının pili bitince",
    "Şarj kablosu temassızlık yapınca",
    "Otobüs şoförü durakta durmayınca",
    "Sinema bileti alırken sıra olunca",
    "Restoranda sipariş yanlış gelince",
    "Hesap öderken 'Alman usulü' yapınca",
    "Düğün konvoyu trafiği kapatınca",
    "Bayram ziyaretinde el öperken",
    "Yaz tatili için bavul hazırlarken",
    "Pasaport kontrolünde sıra beklerken",
    "Havalimanında uçak rötar yapınca",
    "Yeni taşındığım evin anahtarını çevirirken",

    # --- KARIŞIK EKSTRA (25 Adet) ---
    "Romanın en heyecanlı yerinde",
    "Rüyamda uçtuğumu görürken",
    "Falcı kahve fincanına bakınca",
    "Piyango bileti amorti vurunca",
    "Sokak sanatçısı gitar çalarken",
    "Kamp ateşinin başında otururken",
    "Yıldız kaydığında dilek tutarken",
    "Mezuniyet kepini havaya fırlatırken",
    "İlk iş görüşmesine giderken",
    "Ehliyet sınavında park ederken",
    "Doğum günü pastası üflenirken",
    "Yılbaşı gecesi geri sayım yapılırken",
    "Diş ağrısı gece yarısı tutunca",
    "Gözlük camı buğulanınca",
    "Maske takmaktan bunalınca",
    "Dezenfektan kokusu genzimi yakınca",
    "Zoom toplantısında mikrofon açık kalınca",
    "İnternet kotası dolduğunda",
    "Eski fotoğraf albümlerine bakarken",
    "Çocukluk arkadaşımla karşılaşınca",
    "Lunaparkta gondola bindiğimde",
    "Korku filmi izlerken gözümü kapatınca",
    "Kütüphanede sessizliği bozan biri olunca",
    "Otobiyografimi yazmaya başlasam",
    "Issız bir adaya düşseydim"
]

# Üretilen verileri saklayacağımız liste
dataset = []

print(f"Toplam {len(prompts)} adet prompt için üretim başlıyor...\n")

# 2. ADIM: Döngü (Loop)
for prompt in tqdm(prompts, desc="Üretiliyor"):

    # Prompt'u modele uygun formata getir
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Modeli çalıştır (Generation)
    # max_new_tokens: Modelin ne kadar uzun yazacağını belirler
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,   # Çok uzun olmasın, 1-2 cümle yeter
        temperature=0.7,     # Biraz yaratıcı olsun ki hata yapsın
        do_sample=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

    # Çıktıyı yazıya çevir
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Sadece yeni üretilen kısmı ayıklamak istersen (Opsiyonel):
    # generated_part = raw_output.replace(prompt, "").strip()

    # Veriyi listeye ekle
    dataset.append({
        "prompt": prompt,
        "raw_model_output": raw_output  # Bu senin "Bozuk Girdi" verin olacak
    })

# 3. ADIM: Kaydetme (JSON Formatında)
output_file = "ham_veri_seti.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)

print(f"\nİşlem tamamlandı! Veriler '{output_file}' adıyla kaydedildi.")

Toplam 528 adet prompt için üretim başlıyor...



Üretiliyor: 100%|██████████| 528/528 [56:55<00:00,  6.47s/it]


İşlem tamamlandı! Veriler 'ham_veri_seti.json' adıyla kaydedildi.
